# SCT From-Scratch Training
**The core hypothesis:** SCT fails at fine-tuning because rank truncation destroys pretrained structure the model depends on. Training from scratch means the model never had full rank, so it learns to use the available spectral budget from step one.

If loss decreases steadily without hitting the ~4.5 floor we see in fine-tuning, SCT is viable for pre-training.

SmolLM2-1.7B architecture, MLP spectral at rank 32, attention dense. WikiText-103.

**Runtime:** Set to T4 GPU. Expected: ~8 hours for 20K steps. Leave running overnight.

In [ ]:
# Cell 1: Setup
!pip install -q transformers datasets
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: SpectralLinear with from-scratch initialization

def safe_qr(M):
    dev, dt = M.device, M.dtype
    Q, R = torch.linalg.qr(M.float().cpu())
    return (Q * torch.sign(torch.diag(R))).to(dev).to(dt)

class SpectralLinear(nn.Module):
    """W = U diag(s) V^T with Stiefel QR retraction."""
    def __init__(self, in_features, out_features, rank=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = min(rank, min(in_features, out_features))
        # Initialize U, V as random orthonormal
        U = torch.randn(in_features, self.rank)
        V = torch.randn(out_features, self.rank)
        Q_U, R_U = torch.linalg.qr(U)
        Q_V, R_V = torch.linalg.qr(V)
        self.U = nn.Parameter((Q_U * torch.sign(torch.diag(R_U)))[:, :self.rank])
        self.V = nn.Parameter((Q_V * torch.sign(torch.diag(R_V)))[:, :self.rank])
        # Initialize s to match Kaiming scale
        # Dense Kaiming: std = sqrt(2/fan_in), ||W||_F ~ sqrt(fan_out) * std
        # For W = U diag(s) V^T: ||W||_F = ||s||_2
        # So each s_i ~ sqrt(2 * fan_out / (fan_in * rank))
        scale = math.sqrt(2.0 * out_features / (in_features * self.rank))
        self.s = nn.Parameter(torch.full((self.rank,), scale))

    def forward(self, x):
        U = self.U.to(x.dtype)
        s = self.s.to(x.dtype)
        V = self.V.to(x.dtype)
        return (x @ U) * s @ V.T

    @torch.no_grad()
    def retract(self):
        self.U.data = safe_qr(self.U.data)[:, :self.rank]
        self.V.data = safe_qr(self.V.data)[:, :self.rank]

def retract_all(model):
    for m in model.modules():
        if isinstance(m, SpectralLinear):
            m.retract()

print("SpectralLinear (from-scratch init) ready.")

In [ ]:
# Cell 3: Build model from scratch with spectral MLP
from transformers import LlamaConfig, LlamaForCausalLM, AutoTokenizer

RANK = 32
DEVICE = "cuda"

# SmolLM2-1.7B architecture
config = LlamaConfig(
    hidden_size=2048,
    intermediate_size=8192,
    num_hidden_layers=24,
    num_attention_heads=32,
    num_key_value_heads=32,
    vocab_size=49152,
    max_position_embeddings=2048,
    rms_norm_eps=1e-5,
    tie_word_embeddings=True,
    hidden_act="silu",
)

print("Initializing fresh 1.7B model (random weights)...")
model = LlamaForCausalLM(config)

# Convert MLP layers to SpectralLinear
converted = 0
for name, module in model.named_modules():
    for attr in ['gate_proj', 'up_proj', 'down_proj']:
        if hasattr(module, attr):
            linear = getattr(module, attr)
            if isinstance(linear, nn.Linear):
                spec = SpectralLinear(linear.in_features, linear.out_features, RANK)
                setattr(module, attr, spec)
                converted += 1

print(f"Converted {converted} MLP layers to SpectralLinear (rank={RANK})")

# Move to GPU in bfloat16
model = model.to(dtype=torch.bfloat16, device=DEVICE)

# Count params
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
sct_params = sum(p.numel() for m in model.modules() if isinstance(m, SpectralLinear)
                 for p in m.parameters())
dense_equiv = config.num_hidden_layers * (3 * config.hidden_size * config.intermediate_size)

print(f"\nTotal params:      {total_params:>12,}")
print(f"Trainable:         {trainable:>12,}")
print(f"SCT (MLP) params:  {sct_params:>12,}")
print(f"Dense MLP equiv:   {dense_equiv:>12,}")
print(f"MLP compression:   {dense_equiv/sct_params:.1f}x")

mem = torch.cuda.max_memory_allocated() / 1e9
print(f"GPU after model:   {mem:.1f} GB")

In [ ]:
# Cell 4: Load dataset
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B")
tokenizer.pad_token = tokenizer.eos_token

print("Loading WikiText-103...")
dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

# Filter out empty lines and short texts
texts = [t for t in dataset["text"] if len(t.strip()) > 100]
print(f"Filtered texts: {len(texts):,}")

MAX_SEQ_LEN = 256

# Tokenize in chunks (memory efficient)
print("Tokenizing (this takes a few minutes)...")
all_ids = []
for i in range(0, len(texts), 1000):
    batch_texts = texts[i:i+1000]
    enc = tokenizer(batch_texts, truncation=True, max_length=MAX_SEQ_LEN,
                    padding="max_length", return_tensors="pt")
    all_ids.append(enc["input_ids"])
    if (i // 1000) % 20 == 0:
        print(f"  Tokenized {i+1000}/{len(texts)}")

input_ids = torch.cat(all_ids, dim=0)
print(f"\nDataset ready: {input_ids.shape[0]:,} sequences, {MAX_SEQ_LEN} tokens each")
print(f"Total tokens: {input_ids.shape[0] * MAX_SEQ_LEN:,}")

In [ ]:
# Cell 5: Training config

STEPS = 20000         # ~8 hours on T4
BATCH_SIZE = 2        # Fits T4 VRAM
GRAD_ACCUM = 4        # Effective batch = 8
LR = 3e-4             # Standard for from-scratch LLM training
WARMUP_STEPS = 500    # Linear warmup
LOG_EVERY = 200
SAVE_EVERY = 5000     # Checkpoint every 5K steps

# Effective tokens per step: batch * accum * seq = 2 * 4 * 256 = 2048
# Total tokens: 20K * 2048 = ~41M tokens
effective_batch = BATCH_SIZE * GRAD_ACCUM
tokens_per_step = effective_batch * MAX_SEQ_LEN
total_tokens = STEPS * tokens_per_step

print(f"Steps: {STEPS}")
print(f"Effective batch: {effective_batch}")
print(f"Tokens/step: {tokens_per_step:,}")
print(f"Total tokens: {total_tokens:,} ({total_tokens/1e6:.1f}M)")
print(f"LR: {LR} with {WARMUP_STEPS} warmup steps")
print(f"Estimated time: {STEPS * 1.3 / 3600:.1f} hours")

In [ ]:
# Cell 6: Optimizer with linear warmup + cosine decay

optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                              weight_decay=0.1, betas=(0.9, 0.95))

def get_lr(step):
    """Linear warmup then cosine decay to 0.1x."""
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, STEPS - WARMUP_STEPS)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

print(f"Optimizer: AdamW (betas=0.9,0.95, wd=0.1)")
print(f"Schedule: linear warmup {WARMUP_STEPS} steps + cosine decay")
print(f"Peak LR: {LR}, min LR: {LR*0.1}")

In [ ]:
# Cell 7: Training loop

print(f"{'='*70}")
print(f"  SCT FROM-SCRATCH TRAINING")
print(f"  1.7B architecture | Rank {RANK} | WikiText-103 | T4 GPU")
print(f"  {STEPS} steps x {tokens_per_step} tokens/step = {total_tokens/1e6:.1f}M tokens")
print(f"{'='*70}\n")

model.train()
n_samples = input_ids.shape[0]

losses = []
step_times = []
t_start = time.time()
accum_loss = 0.0

for step in range(1, STEPS + 1):
    t0 = time.time()

    # Gradient accumulation
    total_loss = 0.0
    for micro in range(GRAD_ACCUM):
        idx = torch.randint(0, n_samples, (BATCH_SIZE,))
        xb = input_ids[idx].to(DEVICE)
        labels = xb.clone()
        # Mask padding
        labels[labels == tokenizer.pad_token_id] = -100

        outputs = model(input_ids=xb, labels=labels)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item()

    # Clip, step, retract
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    retract_all(model)

    step_time = time.time() - t0
    losses.append({"step": step, "loss": total_loss, "time": step_time,
                   "lr": scheduler.get_last_lr()[0]})
    step_times.append(step_time)

    if step % LOG_EVERY == 0 or step == 1:
        avg_loss = sum(l["loss"] for l in losses[-LOG_EVERY:]) / min(LOG_EVERY, len(losses))
        ppl = math.exp(min(avg_loss, 20))
        avg_time = sum(step_times[-LOG_EVERY:]) / len(step_times[-LOG_EVERY:])
        elapsed = (time.time() - t_start) / 3600
        eta = (STEPS - step) * avg_time / 3600
        cur_lr = scheduler.get_last_lr()[0]
        mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"  Step {step:>6d}/{STEPS}  Loss: {total_loss:.4f}  "
              f"Avg: {avg_loss:.4f}  PPL: {ppl:.1f}  "
              f"LR: {cur_lr:.1e}  "
              f"GPU: {mem:.1f}GB  "
              f"Time: {elapsed:.1f}h  ETA: {eta:.1f}h")

    # Save checkpoint
    if step % SAVE_EVERY == 0:
        ckpt = {"step": step, "losses": losses}
        with open(f"sct_scratch_ckpt_{step}.json", "w") as f:
            json.dump(ckpt, f)
        print(f"  >> Checkpoint saved at step {step}")

total_time = time.time() - t_start
print(f"\nTraining complete in {total_time/3600:.1f} hours.")

In [ ]:
# Cell 8: Results and plot
import matplotlib.pyplot as plt

final_loss = losses[-1]["loss"]
final_ppl = math.exp(min(final_loss, 20))
avg_final = sum(l["loss"] for l in losses[-200:]) / min(200, len(losses))
avg_final_ppl = math.exp(min(avg_final, 20))
peak_mem = torch.cuda.max_memory_allocated() / 1e9

print(f"{'='*70}")
print(f"  FROM-SCRATCH TRAINING RESULTS")
print(f"{'='*70}")
print(f"  Steps completed:           {len(losses)}")
print(f"  Tokens processed:          {len(losses) * tokens_per_step:,}")
print(f"  Final loss (last step):    {final_loss:.4f}")
print(f"  Avg loss (last 200):       {avg_final:.4f}")
print(f"  Final PPL:                 {avg_final_ppl:.1f}")
print(f"  Peak GPU memory:           {peak_mem:.1f} GB")
print(f"  Total time:                {total_time/3600:.1f} hours")
print(f"  MLP compression:           {dense_equiv/sct_params:.1f}x")
print(f"{'='*70}")
print(f"")
print(f"  KEY QUESTION: Is loss still decreasing at step {len(losses)}?")
early = sum(l["loss"] for l in losses[900:1100]) / 200 if len(losses) > 1100 else None
late = sum(l["loss"] for l in losses[-200:]) / 200
if early and late < early:
    improvement = (early - late) / early * 100
    print(f"  YES. Loss improved {improvement:.1f}% from step ~1000 to final.")
    print(f"  From-scratch SCT training is viable. Loss has not plateaued.")
elif early:
    print(f"  NO. Loss stalled. Step ~1000 avg: {early:.4f}, Final avg: {late:.4f}")
print(f"{'='*70}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

step_list = [l["step"] for l in losses]
loss_list = [l["loss"] for l in losses]
lr_list = [l.get("lr", LR) for l in losses]

# Smoothed loss
w = 50
smooth = [sum(loss_list[max(0,i-w+1):i+1]) / (i - max(0,i-w+1) + 1) for i in range(len(loss_list))]

# Loss
axes[0].plot(step_list, loss_list, alpha=0.1, color='#FF5722')
axes[0].plot(step_list, smooth, color='#FF5722', linewidth=2, label=f'SCT from scratch (r={RANK})')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('From-Scratch Training: Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PPL (log scale)
ppl_smooth = [math.exp(min(l, 20)) for l in smooth]
axes[1].plot(step_list, ppl_smooth, color='#FF5722', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('From-Scratch Training: Perplexity')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

# LR schedule
axes[2].plot(step_list, lr_list, color='#2196F3', linewidth=2)
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('LR Schedule')
axes[2].grid(True, alpha=0.3)

fig.text(0.5, 0.01,
         f'SmolLM2-1.7B arch | WikiText-103 | From scratch | '
         f'{torch.cuda.get_device_name(0)} | Rank {RANK} | EctoSpace/SCT',
         ha='center', fontsize=9, color='gray')

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('sct_from_scratch.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved.")

In [ ]:
# Cell 9: Test generation (does it produce anything?

model.eval()
prompts = [
    "The capital of France is",
    "In the year 2024,",
    "The quick brown fox",
    "Machine learning is",
]

print("\nGeneration test (from-scratch model, 20K steps):")
print("="*60)
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=50, do_sample=True,
            temperature=0.8, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {text}")
print("\n" + "="*60)
print("Note: Only ~41M tokens of training. Coherent output not expected.")
print("The metric that matters is whether loss was still decreasing.")

In [ ]:
# Cell 10: Save and download

summary = {
    "experiment": "SCT from-scratch training",
    "model_arch": "SmolLM2-1.7B (LlamaConfig)",
    "initialization": "random (not pretrained)",
    "dataset": "wikitext-103",
    "hardware": torch.cuda.get_device_name(0),
    "rank": RANK,
    "mlp_compression": f"{dense_equiv/sct_params:.1f}x",
    "total_params": total_params,
    "sct_params": sct_params,
    "steps": len(losses),
    "tokens_processed": len(losses) * tokens_per_step,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "effective_batch": effective_batch,
    "seq_len": MAX_SEQ_LEN,
    "lr": LR,
    "warmup": WARMUP_STEPS,
    "final_loss": final_loss,
    "avg_final_loss": avg_final,
    "avg_final_ppl": avg_final_ppl,
    "peak_gpu_gb": peak_mem,
    "total_time_hours": total_time / 3600,
}

with open("sct_from_scratch_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
with open("sct_from_scratch_losses.json", "w") as f:
    json.dump(losses, f)

print("Results saved.")

from google.colab import files
files.download('sct_from_scratch.png')
files.download('sct_from_scratch_summary.json')
files.download('sct_from_scratch_losses.json')
# Download latest checkpoint too
import glob
ckpts = glob.glob('sct_scratch_ckpt_*.json')
for c in ckpts:
    files.download(c)
print("All files downloaded.")